# HistoLente — treino do classificador (Google Colab)

Ambiente de execução → Alterar o tipo de ambiente → **GPU**.

Protocolo: treino e validação no NCT-CRC-HE-100K; **teste final no CRC-VAL-HE-7K** (pacientes diferentes), usado uma única vez.

Salve um token do GitHub (somente leitura do repositório) em *Secrets* do Colab com o nome `GH_TOKEN`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, torchvision; print(torch.__version__, torchvision.__version__, torch.cuda.is_available())

In [ ]:
from google.colab import userdata, drive
drive.mount('/content/drive')
token = userdata.get('GH_TOKEN')
!git clone -q https://{token}@github.com/Umizin/histolente.git /content/histolente
%cd /content/histolente/ml
!git log --oneline -1
!pip install -q scikit-learn==1.5.2 matplotlib==3.9.2

## 1. Dados
Demora alguns minutos (cerca de 12 GB). Os arquivos ficam no disco temporário do Colab.

In [ ]:
!bash scripts/baixar_dados.sh /content/dados

## 2. Divisão treino/validação (estratificada, semente 42)

In [ ]:
!python -m histolente_ml dividir --raiz /content/dados/NCT-CRC-HE-100K --saida splits

## 3. Treino
Comece pela ResNet18 (mais leve em CPU); repita com `efficientnet_b0` e `resnet50` para a comparação de backbones.

In [ ]:
BACKBONE = 'resnet18'
RUN = f'/content/drive/MyDrive/histolente/runs/{BACKBONE}'
!python -m histolente_ml treinar --raiz /content/dados/NCT-CRC-HE-100K --splits splits --backbone {BACKBONE} --epocas 10 --lote 128 --workers 2 --saida {RUN}

## 4. Avaliação no teste independente (CRC-VAL-HE-7K)

In [ ]:
REL = f'/content/drive/MyDrive/histolente/reports/{BACKBONE}'
!python -m histolente_ml avaliar --modelo {RUN}/melhor.pt --backbone {BACKBONE} --raiz /content/dados/CRC-VAL-HE-7K --saida {REL} --workers 2

## 5. Exportar para o serviço de inferência
Baixe a pasta `models/` do Drive e coloque em `ml/models/` no computador (os `.pt` não vão para o Git).

In [ ]:
!python -m histolente_ml exportar --modelo {RUN}/melhor.pt --backbone {BACKBONE} --versao 0.1.0 --treino {RUN}/treino.json --metricas {REL}/metricas.json --destino /content/drive/MyDrive/histolente/models

## 6. Copiar os relatórios para o repositório
Copie `metricas.json`, `matriz_confusao.png` e `treino.json` para `ml/reports/<backbone>/` e faça um PR. A latência para o RNF01 **não** é medida aqui (GPU do Colab ≠ máquina de uso): rode `python -m histolente_ml latencia` na máquina de referência, ociosa.